# Desafio Transparência - Análise Consolidada (Camada Gold)
Este notebook realiza a extração, transformação e análise de dados sobre diárias, passagens e viagens corporativas. Os resultados são apresentados em DataFrames e exportados como imagens estáticas em alta resolução (`.png`) na pasta `graficos_png/`.

In [ ]:
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

# Configurar localidade global do Plotly para formato brasileiro (Ponto para milhar, Vírgula para decimal)
pio.templates[pio.templates.default].layout.separators = ',.'

# Garantir que a pasta de gráficos exista
os.makedirs('graficos_png', exist_ok=True)

# Definir renderizador estático como padrão para visualização limpa e exportação
pio.renderers.default = 'png'

# Ajustar caminhos do sistema para incluir a pasta 'scripts'
diretorio_atual = os.getcwd()
diretorio_scripts = os.path.abspath(os.path.join(diretorio_atual, 'scripts'))

if diretorio_scripts not in sys.path:
    sys.path.append(diretorio_scripts)

# Conexão com o banco de dados
try:
    import banco
    conn = banco.conectar()
    print('Conexão estabelecida com sucesso!')
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'O arquivo "banco.py" não foi encontrado dentro da pasta "scripts". '
        'Verifique a estrutura do seu projeto.'
    )

## 1. Top 5 Órgãos com Maior Custo Total
**Pergunta de Negócio:** Quais são os 5 órgãos públicos que acumularam o maior valor em gastos com viagens e diárias?

In [ ]:
query_1 = 'SELECT * FROM gold_vw_custo_por_orgao;'
df1 = pd.read_sql(query_1, conn)

col_orgao = df1.columns[0]
col_custo = df1.columns[1]

df1[col_custo] = pd.to_numeric(df1[col_custo], errors='coerce')
df1_sorted = df1.sort_values(col_custo, ascending=False).head(5)
df1_sorted = df1_sorted.sort_values(col_custo, ascending=True)

fig1 = px.bar(
    df1_sorted,
    x=col_custo,
    y=col_orgao,
    orientation='h',
    title='Top 5 Órgãos por Custo Total de Viagens'
)

max_val1 = df1_sorted[col_custo].max()
fig1.update_traces(
    textposition='outside',
    texttemplate='R$ %{x:,.2f}',
    cliponaxis=False
)
fig1.update_layout(
    margin=dict(l=50, r=150, t=60, b=50), 
    xaxis=dict(range=[0, max_val1 * 1.25], title='Custo Total (R$)'),
    yaxis=dict(title='', automargin=True),
    uniformtext=dict(mode='hide', minsize=10)
)

fig1.write_image('graficos_png/1_maior_custo_total_orgao.png', scale=2)
fig1.show()

## 2. Top 3 Destinos mais Frequentes
**Pergunta de Negócio:** Quais os destinos mais frequentados nas viagens registradas?

In [ ]:
query_2 = 'SELECT * FROM gold_vw_destinos_frequentes;'
df2 = pd.read_sql(query_2, conn)

col_destino = df2.columns[0]
col_qtd = df2.columns[1]

# Forçar a coluna de contagem para tipo numérico
df2[col_qtd] = pd.to_numeric(df2[col_qtd], errors='coerce')
df2_sorted = df2.sort_values(col_qtd, ascending=False).head(3)

fig2 = px.bar(
    df2_sorted,
    x=col_destino,
    y=col_qtd,
    title='Top 3 Destinos Mais Frequentes'
)

max_val2 = df2_sorted[col_qtd].max()
fig2.update_traces(
    textposition='outside',
    texttemplate='%{y:,.0f}',
    cliponaxis=False
)
fig2.update_layout(
    margin=dict(l=60, r=60, t=80, b=60),
    yaxis=dict(range=[0, max_val2 * 1.15], title='Quantidade de Viagens'),
    xaxis=dict(title='Destino', automargin=True)
)

fig2.write_image('graficos_png/2_custo_medio_destino.png', scale=2)
fig2.show()

## 3. Viagem de Maior Duração
**Pergunta de Negócio:** Qual foi o registro de viagem corporativa com o maior número de dias contínuos?

In [ ]:
query_3 = 'SELECT * FROM fato_viagem;'
df3 = pd.read_sql(query_3, conn)

col_duracao = [c for c in df3.columns if 'duracao' in c.lower() or 'dias' in c.lower()][0]
df3[col_duracao] = pd.to_numeric(df3[col_duracao], errors='coerce')

df3_sorted = df3.sort_values(col_duracao, ascending=False).head(1)
display(df3_sorted)

## 4. Evolução dos Gastos Mensais
**Pergunta de Negócio:** Como se comportou a evolução financeira total dos gastos ao longo dos meses mapeados?

In [ ]:
query_4 = 'SELECT * FROM gold_vw_gastos_mensais;'
df4 = pd.read_sql(query_4, conn)

col_mes = df4.columns[0]
col_gasto = df4.columns[1]

df4[col_gasto] = pd.to_numeric(df4[col_gasto], errors='coerce')
df4_sorted = df4.sort_values(col_mes, ascending=True)

fig4 = px.line(
    df4_sorted,
    x=col_mes,
    y=col_gasto,
    title='Evolução Mensal de Gastos (Camada Gold)',
    markers=True
)
fig4.update_layout(
    margin=dict(l=60, r=60, t=80, b=60),
    xaxis=dict(title='Período'),
    yaxis=dict(title='Gasto Total (R$)')
)

fig4.write_image('graficos_png/4_valor_medio_pagamento.png', scale=2)
fig4.show()

## 5. Distribuição de Meios de Transporte nos Trechos
**Pergunta de Negócio:** Qual a proporção do uso de meios de transporte nos trechos realizados na camada Silver?

In [ ]:
query_5 = 'SELECT * FROM silver_trecho;'
df5 = pd.read_sql(query_5, conn)

col_meio = [c for c in df5.columns if 'meio' in c.lower() or 'transporte' in c.lower()][0]

df5_contagem = df5[col_meio].value_counts().reset_index()
df5_contagem.columns = [col_meio, 'total_trechos']
df5_contagem = df5_contagem[df5_contagem[col_meio].str.strip() != '']

fig5 = px.pie(
    df5_contagem,
    values='total_trechos',
    names=col_meio,
    title='Distribuição dos Meios de Transporte Utilizados',
    hole=0.4
)
fig5.update_traces(textinfo='percent+label', texttemplate='%{label}: %{percent:.1%}')
fig5.update_layout(margin=dict(l=50, r=50, t=60, b=50))

fig5.write_image('graficos_png/5_meio_transporte_mais_usado.png', scale=2)
fig5.show()

## 6. Frequência de Trechos por UF de Destino
**Pergunta de Negócio:** Quais estados da federação (UF) concentram o maior volume de trechos na base Silver?

In [ ]:
query_6 = 'SELECT * FROM silver_trecho;'
df6 = pd.read_sql(query_6, conn)

col_uf = [c for c in df6.columns if 'uf' in c.lower() and 'dest' in c.lower()][0]

df6_contagem = df6[col_uf].value_counts().reset_index()
df6_contagem.columns = [col_uf, 'frequencia']
df6_contagem = df6_contagem[df6_contagem[col_uf].str.strip() != '']

fig6 = px.bar(
    df6_contagem,
    x=col_uf,
    y='frequencia',
    title='Frequência de Trechos por UF de Destino'
)

max_val6 = df6_contagem['frequencia'].max()
fig6.update_traces(
    textposition='outside',
    texttemplate='%{y:,.0f}',
    cliponaxis=False
)
fig6.update_layout(
    margin=dict(l=50, r=50, t=80, b=100),
    xaxis=dict(tickangle=-45, title='UF de Destino', automargin=True),
    yaxis=dict(range=[0, max_val6 * 1.15], title='Total de Trechos')
)

fig6.write_image('graficos_png/6_uf_destino_frequencia.png', scale=2)
fig6.show()

## 7. Órgão Superior com Maior Volume de Pagamentos Realizados
**Pergunta de Negócio:** Qual órgão acumulou o maior montante a partir da tabela analítica consolidada de pagamentos?

In [ ]:
query_7 = 'SELECT * FROM gold_orgao_pagamentos;'
df7 = pd.read_sql(query_7, conn)

col_orgao_pago = df7.columns[0]
col_total_pago = df7.columns[1]

df7[col_total_pago] = pd.to_numeric(df7[col_total_pago], errors='coerce')
df7_sorted = df7.sort_values(col_total_pago, ascending=False).head(5)
df7_sorted = df7_sorted.sort_values(col_total_pago, ascending=True)

fig7 = px.bar(
    df7_sorted,
    x=col_total_pago,
    y=col_orgao_pago,
    orientation='h',
    title='Top Órgãos por Total Pago'
)

max_val7 = df7_sorted[col_total_pago].max()
fig7.update_traces(
    textposition='outside',
    texttemplate='R$ %{x:,.2f}',
    cliponaxis=False
)
fig7.update_layout(
    margin=dict(l=50, r=150, t=60, b=50),
    xaxis=dict(range=[0, max_val7 * 1.25], title='Total Pago (R$)'),
    yaxis=dict(title='', automargin=True),
    uniformtext=dict(mode='hide', minsize=10)
)

fig7.write_image('graficos_png/7_orgao_que_mais_pagou.png', scale=3)
fig7.show()